# Merge campaigns by country

Set `country_name`, review the matching folders, then run the merge cell.
Folder matching is case-insensitive and includes the country name anywhere in the
folder name. The destination country folder and folders ending in `_merged` are excluded.

The new dataset is saved as `<country>/<country>_part_all.gzip.parquet`.
For each campaign, use its combined raw Parquet if available; otherwise use local
parts or Parquet files inside ZIP archives. All rows are retained, with a
`source_campaign` column. Missing columns become nulls and stored pandas indexes
are discarded. This merges raw data only. Existing destination files are replaced
on rerun. Use the `backbone_env_v2` environment.

In [1]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

country_name = "China"

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "data/Labeled_Datasets").is_dir()
)
datasets_directory = project_root / "data/Labeled_Datasets"
country_name = country_name.strip()
if not country_name or Path(country_name).name != country_name or country_name in {".", ".."}:
    raise ValueError("Enter a country name without directory separators.")

new_dataset_name = country_name
output_path = (
    datasets_directory / new_dataset_name
    / f"{new_dataset_name}_part_all.gzip.parquet"
)
campaign_folders = sorted(
    folder for folder in datasets_directory.iterdir()
    if folder.is_dir()
    and country_name.casefold() in folder.name.casefold()
    and folder.name.casefold() != new_dataset_name.casefold()
    and not folder.name.casefold().endswith("_merged")
)
if not campaign_folders:
    raise ValueError(f"No campaign folders match {country_name!r}.")

display(pd.DataFrame({"campaign": [folder.name for folder in campaign_folders]}))
print(f"Destination: {output_path.relative_to(project_root)}")

Destination: data/Labeled_Datasets/China/China_part_all.gzip.parquet


,campaign
0,China_1
1,China_2


In [2]:
def find_campaign_files(folder):
    """Find raw Parquet inputs without including both combined files and parts.

    Args:
        folder: Campaign directory containing raw files or ZIP archives.

    Returns:
        A list of (path, archive_member) pairs; archive_member is None for local files.
    """
    combined_path = folder / f"{folder.name}_part_all.gzip.parquet"
    if combined_path.is_file():
        return [(combined_path, None)]
    parts = sorted(folder.glob("*.parquet"))
    if parts:
        return [(path, None) for path in parts]
    files = []
    for archive_path in sorted(folder.glob("*.zip")):
        with ZipFile(archive_path) as archive:
            members = sorted(name for name in archive.namelist() if name.endswith(".parquet"))
            combined_members = [name for name in members if "part_all" in Path(name).name]
            files.extend((archive_path, name) for name in (combined_members or members))
    if not files:
        raise FileNotFoundError(f"No raw Parquet files found in {folder}")
    return files


def open_parquet(path, archive_member):
    """Open a local Parquet file or one member of a ZIP archive.

    Args:
        path: Parquet or ZIP path.
        archive_member: Parquet member name, or None for a local file.

    Returns:
        A ParquetFile readable in batches.
    """
    if archive_member is None:
        return pq.ParquetFile(path)
    with ZipFile(path) as archive:
        return pq.ParquetFile(BytesIO(archive.read(archive_member)))


inputs = [
    (folder.name, path, member)
    for folder in campaign_folders
    for path, member in find_campaign_files(folder)
]
schemas = [open_parquet(path, member).schema_arrow.remove_metadata()
           for _, path, member in inputs]
output_schema = pa.schema([
    field for field in pa.unify_schemas(schemas)
    if not field.name.startswith("__index_level_") and field.name != "source_campaign"
]).append(pa.field("source_campaign", pa.string()))

## Create the merged dataset

Run this cell to write the new dataset. Batches limit memory use; one archived
Parquet part is decompressed at a time. The completed file replaces the destination
only after its row count is verified.

In [3]:
output_path.parent.mkdir(parents=True, exist_ok=True)
temporary_path = output_path.with_suffix(".tmp")
row_counts = []

with pq.ParquetWriter(temporary_path, output_schema, compression="gzip") as writer:
    for campaign, path, member in inputs:
        parquet = open_parquet(path, member)
        for batch in parquet.iter_batches(batch_size=25_000):
            table = pa.Table.from_batches([batch])
            columns = []
            for field in output_schema:
                if field.name == "source_campaign":
                    values = pa.array([campaign] * len(table), type=pa.string())
                elif field.name in table.column_names:
                    values = table[field.name].cast(field.type)
                else:
                    values = pa.nulls(len(table), type=field.type)
                columns.append(values)
            writer.write_table(pa.Table.from_arrays(columns, schema=output_schema))
        row_counts.append({"campaign": campaign, "rows": parquet.metadata.num_rows})
        print(f"{campaign}: {parquet.metadata.num_rows:,} rows from {member or path.name}", flush=True)

summary = pd.DataFrame(row_counts).groupby("campaign", sort=False)[["rows"]].sum()
assert pq.read_metadata(temporary_path).num_rows == summary["rows"].sum()
temporary_path.replace(output_path)

summary.loc["Total"] = summary.sum()
summary_directory = project_root / "result" / new_dataset_name
summary_directory.mkdir(parents=True, exist_ok=True)
summary.to_csv(summary_directory / "campaign_merge_summary.csv")
display(summary.style.format("{:,}"))
print(f"Saved: {output_path.relative_to(project_root)}")

China_1: 50,000 rows from China_1/China_1_part_1.gzip.parquet
China_1: 50,000 rows from China_1/China_1_part_10.gzip.parquet
China_1: 50,000 rows from China_1/China_1_part_11.gzip.parquet
China_1: 50,000 rows from China_1/China_1_part_12.gzip.parquet
China_1: 50,000 rows from China_1/China_1_part_13.gzip.parquet
China_1: 50,000 rows from China_1/China_1_part_14.gzip.parquet
China_1: 50,000 rows from China_1/China_1_part_15.gzip.parquet
China_1: 50,000 rows from China_1/China_1_part_16.gzip.parquet
China_1: 50,000 rows from China_1/China_1_part_17.gzip.parquet
China_1: 50,000 rows from China_1/China_1_part_18.gzip.parquet
China_1: 50,000 rows from China_1/China_1_part_19.gzip.parquet
China_1: 50,000 rows from China_1/China_1_part_2.gzip.parquet
China_1: 50,000 rows from China_1/China_1_part_20.gzip.parquet
China_1: 50,000 rows from China_1/China_1_part_21.gzip.parquet
China_1: 50,000 rows from China_1/China_1_part_22.gzip.parquet
China_1: 50,000 rows from China_1/China_1_part_23.gzip.pa

,rows
campaign,
China_1,"3,670,754"
China_2,"3,414,054"
Total,"7,084,808"
